# **GENETIC ALGORITHM**

This is a genetic algorithm implementation from scratch 

In [2]:
import random
import numpy as np
import pandas as pd

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df = pd.read_csv("/content/drive/MyDrive/Student_Performance.csv")

In [5]:
df = df.drop("Extracurricular Activities", axis=1)

In [6]:
X = df.drop('Performance Index', axis=1)
Y = df['Performance Index']

In [7]:
def initialize_poblation(size=100):
    random_seed = 42
    np.random.seed(random_seed)
    random.seed(42)
    poblacion = np.random.uniform(-20,20, size=(size, 4))
    poblacion = pd.DataFrame(poblacion)
    poblacion = poblacion.rename(columns={0: "theta_0", 1: "theta_1", 2: "theta_2", 3: "theta_3"})
    poblacion["error"] = 0
    return poblacion

In [108]:
def fitness( individuos: pd.DataFrame, X: pd.DataFrame, Y: pd.Series ):
    X = X.astype(float)
    Y = Y.astype(float).values
    resultados_error = []
    individuos = individuos.drop('error', axis=1)
    # error = individuos['error']

    for _, individuo in individuos.iterrows():

        # Extraemos los genes
        theta = individuo.values.astype(float)

        y_pred = X.values @ theta
        error = np.mean((Y - y_pred) ** 2)
        resultados_error.append(error)
    
    error = np.array(resultados_error)
    individuos = individuos.copy()
    individuos["error"] = error

    return individuos


In [101]:
def best_50(poblacion: pd.DataFrame):
    ordered_poblation = poblacion.sort_values(by="error", ascending=True)
    top50= ordered_poblation.head(50)
    return top50

In [102]:
def cruce(individuos: pd.DataFrame, gamma: float = 0.5):
  poblacion = individuos.copy()
  theta_0_values = individuos["theta_0"].values.flatten()
  theta_1_values = individuos["theta_1"].values.flatten()
  theta_2_values = individuos["theta_2"].values.flatten()
  theta_3_values = individuos["theta_3"].values.flatten()

  for i in range(0, len(theta_1_values),2):
    if random.random() <= gamma:
      theta_0_values[i], theta_0_values[i + 1] = theta_0_values[i + 1], theta_0_values[i]
      theta_2_values[i], theta_2_values[i + 1] = theta_2_values[i + 1], theta_2_values[i]
    else:
      theta_1_values[i], theta_1_values[i + 1] = theta_1_values[i + 1], theta_1_values[i]
      theta_3_values[i], theta_3_values[i + 1] = theta_3_values[i + 1], theta_3_values[i]

  poblacion["theta_0"] = theta_0_values
  poblacion["theta_1"] = theta_1_values
  poblacion["theta_2"] = theta_2_values
  poblacion["theta_3"] = theta_3_values

  nueva_poblacion = pd.concat([individuos, poblacion], ignore_index = False)
  nueva_poblacion["error"] = 0

  return nueva_poblacion

In [14]:
def mutacion(poblacion: pd.DataFrame, prob_mutacion: float = 0.25):
    poblacion = poblacion.drop('error', axis=1)
    for idx,_ in poblacion.iterrows():
        for col in poblacion.columns:
            if random.random() <= prob_mutacion:
                poblacion.loc[idx, col] += random.uniform(-2, 2)
    
    poblacion["error"] = 0
    return poblacion

In [103]:
def genetic_algorithm(generations: int, X: pd.DataFrame, Y: pd.Series, poblation_size: int = 100):
    poblation = initialize_poblation(size=poblation_size)
    poblation = fitness(
        X = X,
        Y = Y,
        individuos = poblation
    )

    for i in range(generations):
        top50 = best_50(poblacion= poblation)

        new_generation = cruce( 
            individuos= top50 
        )

        new_generation = mutacion(
            poblacion = new_generation
        )

        poblation = new_generation

        poblation = fitness(
            X = X,
            Y = Y,
            individuos = poblation
        )

        # print(f'[INFO] Generación {i+1}')
    
    best_gen = poblation.sort_values(by='error').iloc[0]

    return best_gen

In [113]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [115]:
gen  = genetic_algorithm(
    generations = 100,
    X = X_scaled,
    Y = Y
)

gen

AttributeError: 'numpy.ndarray' object has no attribute 'values'

In [110]:
gen  = genetic_algorithm(
    generations = 500,
    X = X,
    Y = Y
)

gen 

,75
theta_0,2.408478e+02
theta_1,1.250450e+02
theta_2,2.718327e+02
theta_3,-8.999703e+01
error,1.305523e+08


In [112]:
gen  = genetic_algorithm(
    generations = 1000,
    X = X_scaled,
    Y = Y
)

gen

AttributeError: 'numpy.ndarray' object has no attribute 'values'

In [34]:
import sklearn
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X, Y)
lin_reg.intercept_, lin_reg.coef_

(np.float64(-33.76372609079468),
 array([2.85342921, 1.01858354, 0.47633298, 0.1951983 ]))

In [80]:
lin_reg.predict(X.loc[6].to_numpy().reshape(1,-1))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([64.11973141])

In [88]:
xs = X.loc[6].to_numpy()

gen = gen.drop('error')

pred = xs @ gen + lin_reg.coef_
pred

KeyError: "['error'] not found in axis"

In [92]:
gen

,75
theta_0,444.344054
theta_1,49.439102
theta_2,233.118048
theta_3,-8.254565


In [91]:
xs = X.loc[6].to_numpy()

# gen = gen.drop('error')

pred = xs @ gen + lin_reg.intercept_
pred

np.float64(7801.761954583217)